# From local integration to own global assembly

We use the same own finite element space in three stages: symbolic forms, own C++ integrators with the framework's element loop, and own integrators with an own serial element loop.

`myIntegrator.hpp` contains the quadrature loops. `myAssembling.hpp` contains the matrix graph and the local-to-global sums.

The triangular examples use `P1TriangleFESpace`. The quadrilateral examples use `HighOrderFESpace` at degree one. The C++ classes `LaplaceIntegrator`, `MassIntegrator`, `BoundaryMassIntegrator`, `SourceIntegrator` and `BoundarySourceIntegrator` each implement one weak-form term. The Python factories are `m.Laplace`, `m.Mass`, `m.BoundaryMass`, `m.Source` and `m.BoundarySource`.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ngsolve import *
from teaching_utils import load_module, square_mesh, max_diameter, errors
m = load_module()

## From reference gradients to one local stiffness matrix

The reference triangle has vertices $(1,0),(0,1),(0,0)$. Write the affine map as $F_T(\widehat x)=J_T\widehat x+p_2$. Storing one gradient per row gives

$$
\begin{align*}
D_x=D_{\mathrm{ref}}J_T^{-1},\qquad A_T=\sum_q \widehat w_q\,|\det J_T|\,D_xD_x^\top.
\end{align*}
$$

For $P_1$ diffusion with coefficient one, the integrand is constant. A single reference point of weight $1/2$ integrates it exactly. The code displays every factor on the first physical triangle and compares this calculation with the C++ quadrature loop. The matrix is positive semidefinite because constants have zero gradient, however, the global Dirichlet conditions remove this kernel.

In [ ]:
mesh = square_mesh(4)
fes = m.P1TriangleFESpace(mesh,dirichlet=".*")
cell = next(iter(mesh.Elements()))
ei = ElementId(VOL,cell.nr)
points = np.array([mesh.vertices[vertex.nr].point[:2] for vertex in cell.vertices])
J = np.column_stack((points[0]-points[2],points[1]-points[2]))
D_ref = np.array(fes.GetFE(ei).CalcDShape(1/3,1/3))
D_x = D_ref@np.linalg.inv(J)
quadrature_matrix = 0.5*abs(np.linalg.det(J))*(D_x@D_x.T)
local_matrix = np.array(m.Laplace(CF(1)).CalcElementMatrix(fes.GetFE(ei),mesh.GetTrafo(ei)))
print(f"Jacobian J:\n{J}")
print(f"Reference gradients (rows):\n{D_ref}")
print(f"Physical gradients (rows):\n{D_x}")
print(f"One-point quadrature matrix:\n{quadrature_matrix}")
print(f"Difference from the C++ matrix: {np.linalg.norm(quadrature_matrix-local_matrix):.3e}")

## Three paths to the same discrete system

Take $u=x(1-x)y(1-y)$ and $f=2[x(1-x)+y(1-y)]$. Then $-\Delta u=f$ and $u=0$ on the boundary. The known function will also provide a meaningful discretization error.

All three paths below use the **same own basis**. Entrywise matrix comparisons are valid for these paths. Comparisons with built-in higher-order `H1` should instead compare the represented functions, since its basis differs.

In [ ]:
exact = x*(1-x)*y*(1-y)
source = 2*(x*(1-x)+y*(1-y))
u,v = fes.TnT()

# 1. Own space; symbolic integration and framework assembly.
a_symbolic = BilinearForm(grad(u)*grad(v)*dx).Assemble()
f_symbolic = LinearForm(source*v*dx(bonus_intorder=2)).Assemble()

# 2. Own space and integrators; framework assembly.
a_integrator = BilinearForm(fes)
a_integrator += m.Laplace(CF(1))
a_integrator.Assemble()
f_integrator = LinearForm(fes)
f_integrator += m.Source(source)
f_integrator.Assemble()

# 3. Own space, integrators and global element loops.
A = m.AssembleMatrix(fes,[m.Laplace(CF(1))])
b = m.AssembleVector(fes,[m.Source(source)])
from scipy.sparse import csr_matrix

def sparse_matrix(matrix):
    rows,columns,values = matrix.COO()
    return csr_matrix((values,(rows,columns)),shape=(matrix.height,matrix.width))

symbolic_matrix = sparse_matrix(a_symbolic.mat)
for name,matrix,vector in [("own integrators",a_integrator.mat,f_integrator.vec),
                           ("own global loops",A,b)]:
    matrix_difference = np.linalg.norm((sparse_matrix(matrix)-symbolic_matrix).data)
    vector_difference = Norm(vector-f_symbolic.vec)
    print(f"{name}: matrix difference {matrix_difference:.3e}, load difference {vector_difference:.3e}")

uh = GridFunction(fes)
uh.vec.data = A.Inverse(fes.FreeDofs(),inverse="sparsecholesky")*b
xs = np.linspace(.001,.999,101)
plt.figure(figsize=(6,3.5))
plt.plot(xs,[exact(mesh(t,.5)) for t in xs],label="exact solution")
plt.plot(xs,[uh(mesh(t,.5)) for t in xs],label="own assembly, P1")
plt.xlabel("x at y=0.5"); plt.ylabel("u")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## Boundary cells, lifting, and mixed conditions

To make the boundary assembly visible, use a $Q_1$ mesh and the exactly representable function $u=1+x+xy$. For $-\Delta u+u=f$ we have $f=u$. Impose

- left: $u=1$,
- right: $\partial_nu+2u=5+3y$,
- top: $\partial_nu=x$,
- bottom: $\partial_nu=-x$.

The supplied boundary integrators use segment basis functions and arclength measure. The correction solve acts only on free coefficients. Setting the boundary values and selecting `dirichlet="left"` are separate steps.

In [ ]:
mixed_mesh = square_mesh(3,quadrilateral=True)
mixed_fes = m.HighOrderFESpace(mixed_mesh,order=1,dirichlet="left")
mixed_exact = 1+x+x*y
right = mixed_mesh.Boundaries("right")
top = mixed_mesh.Boundaries("top")
bottom = mixed_mesh.Boundaries("bottom")
A_mixed = m.AssembleMatrix(mixed_fes,[
    m.Laplace(CF(1)),m.Mass(CF(1)),m.BoundaryMass(CF(2),definedon=right)])
b_mixed = m.AssembleVector(mixed_fes,[
    m.Source(mixed_exact),m.BoundarySource(5+3*y,definedon=right),
    m.BoundarySource(x,definedon=top),m.BoundarySource(-x,definedon=bottom)])
uh_mixed = GridFunction(mixed_fes)
uh_mixed.Set(CF(1),definedon=mixed_mesh.Boundaries("left"))
rhs = b_mixed.CreateVector(); rhs.data = b_mixed-A_mixed*uh_mixed.vec
uh_mixed.vec.data += A_mixed.Inverse(mixed_fes.FreeDofs(),inverse="sparsecholesky")*rhs
ys = np.linspace(.001,.999,31)
plt.figure(figsize=(6,3.5))
plt.plot(ys,[mixed_exact(mixed_mesh(.6,t)) for t in ys],label="exact bilinear function")
plt.plot(ys,[uh_mixed(mixed_mesh(.6,t)) for t in ys],"o",label="own mixed-boundary assembly")
plt.xlabel("y at x=0.6"); plt.ylabel("u")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## Why a single quadrature point is insufficient for $Q_1$ diffusion

For $\widehat v=(1-2\xi)(1-2\eta)$, the gradient vanishes at the cell center. Center-point integration therefore gives an additional zero-energy mode. Compare the spectra of the two local matrices below. One zero eigenvalue is expected from constants. An extra zero eigenvalue indicates underintegration.

In [ ]:
quad_mesh = square_mesh(1,quadrilateral=True)
quad_fes = m.HighOrderFESpace(quad_mesh,order=1)
quad_ei = ElementId(VOL,0)
underintegration_spectra = {}
for name,qorder in [("center point",1),("sufficient rule",4)]:
    element_matrix = np.array(m.Laplace(CF(1),quadrature_order=qorder).CalcElementMatrix(
        quad_fes.GetFE(quad_ei),quad_mesh.GetTrafo(quad_ei)))
    eigenvalues = np.linalg.eigvalsh(element_matrix)
    underintegration_spectra[name] = eigenvalues.tolist()
    print(name, np.round(eigenvalues,6))

## Actual errors and convergence

Refine the triangular mesh and assemble the systems with the own C++ loops. The plot uses the maximum **actual element diameter**. The $H^1$ error includes both the function and gradient errors.

For this smooth solution and $P_1$ elements, expect first order in $H^1$ and second order in $L^2$.

In [ ]:
convergence = []
for n in [4,8,16,32]:
    mesh_h = square_mesh(n)
    fes_h = m.P1TriangleFESpace(mesh_h,dirichlet=".*")
    A_h = m.AssembleMatrix(fes_h,[m.Laplace(CF(1))])
    b_h = m.AssembleVector(fes_h,[m.Source(source)])
    uh_h = GridFunction(fes_h)
    uh_h.vec.data = A_h.Inverse(fes_h.FreeDofs(),inverse="sparsecholesky")*b_h
    convergence.append(dict(n=n,h=max_diameter(mesh_h),ndof=fes_h.ndof,
                            **errors(mesh_h,uh_h,exact)))
plt.figure(figsize=(6,4))
for norm in ["L2","H1"]:
    plt.loglog([row["h"] for row in convergence],[row[norm] for row in convergence],"o-",label=norm)
    old,new = convergence[-2:]
    rate = np.log(old[norm]/new[norm])/np.log(old["h"]/new["h"])
    print(f"last {norm} rate: {rate:.3f}")
plt.xlabel("maximum element diameter h"); plt.ylabel("error")
plt.legend(); plt.grid(True,which="both",alpha=.3); plt.tight_layout(); plt.show()